# How X picked 1 of 4 notes
- Tweet: "SAUDI IS BURNING"
- 4 notes, only Note 4 was shown
- Same steps as the diagram

In [1]:
import glob

import pandas as pd

pd.set_option("display.max_colwidth", 80)

# ids are 19 digits, read them as text so pandas doesn't round them
id_columns = {"noteId": str, "tweetId": str, "noteAuthorParticipantId": str, "raterParticipantId": str, "participantId": str}

saudi_folder = "dataset/tweet_2109031558967615703/"
note_names = ["Note 1", "Note 2", "Note 3", "Note 4"]

In [2]:
# the tweet's files
combined = pd.read_csv(saudi_folder + "combined.csv", dtype=id_columns)
notes = pd.read_csv(saudi_folder + "notes.csv", dtype=id_columns)
note_status = pd.read_csv(saudi_folder + "note_status.csv", dtype=id_columns)
ratings = pd.read_csv(saudi_folder + "ratings.csv", dtype=id_columns)

print("notes:", len(notes))
print("ratings:", len(ratings))
print("raters:", ratings["raterParticipantId"].nunique())

notes: 4
ratings: 456
raters: 263


In [3]:
# the tweet
tweet_time = pd.to_datetime(combined["tweet_created_at"][0], utc=True)
print("tweet:", combined["tweet_text"][0])
print("posted:", tweet_time)

tweet: SAUDI IS BURNING
posted: 2026-10-10 21:21:22+00:00


## 1. The 4 notes
- `writer_enrollmentState = apiEarnedIn` means an AI writer

In [4]:
# minutes between the tweet and each note
note_time = pd.to_datetime(notes["createdAtMillis"], unit="ms", utc=True)
notes["minutes_after_tweet"] = (note_time - tweet_time).dt.total_seconds() // 60

notes[["note_label", "writer_type", "writer_enrollmentState", "minutes_after_tweet", "summary"]]

,note_label,writer_type,writer_enrollmentState,minutes_after_tweet,summary
0,Note 1,Human,earnedIn,15.0,Mario claims saudi arabia is burning after an attack on kkia this is poor j...
1,Note 2,AI,apiEarnedIn,48.0,Saudi Arabia faces targeted Houthi attacks on Riyadh airport causing localiz...
2,Note 3,AI,apiEarnedIn,118.0,Saudi Arabia is not &quot;burning.&quot; Official reports detail a specific ...
3,Note 4,Human,earnedIn,119.0,“Saudi is burning” misleadingly implies nationwide fires. Saudi authoritie...


In [5]:
# every note ever written on X (5 files, about a minute)
note_parts = []
for file in sorted(glob.glob("data/notes/notes-*.tsv")):
    one_file = pd.read_csv(file, sep="\t", usecols=["noteAuthorParticipantId"], dtype=id_columns)
    note_parts.append(one_file)
all_notes = pd.concat(note_parts)

notes_per_writer = all_notes["noteAuthorParticipantId"].value_counts()
print("notes on X:", len(all_notes))

notes on X: 3190046


In [6]:
# AI writers, most active first
users = pd.read_csv("data/userEnrollment-00000.tsv", sep="\t", usecols=["participantId", "enrollmentState"], dtype=id_columns)
ai_writers = set(users.loc[users["enrollmentState"] == "apiEarnedIn", "participantId"])

ai_ranking = list(notes_per_writer[notes_per_writer.index.isin(ai_writers)].index)
print("AI writers:", len(ai_ranking))

AI writers: 36


In [7]:
# where our 4 writers stand
for _, note in notes.iterrows():
    writer = note["noteAuthorParticipantId"]
    if writer in ai_ranking:
        print(note["note_label"], "| AI    | rank", ai_ranking.index(writer) + 1, "| notes:", notes_per_writer[writer])
    else:
        print(note["note_label"], "| human | notes:", notes_per_writer[writer])

Note 1 | human | notes: 3
Note 2 | AI    | rank 2 | notes: 45737
Note 3 | AI    | rank 1 | notes: 46308
Note 4 | human | notes: 1


- Notes 2 and 3: the 2 most active AI writers on X
- Note 4: a person's first note

## 2. The ratings

In [8]:
# ratings and % Helpful for each note
for name in note_names:
    levels = ratings.loc[ratings["note_label"] == name, "helpfulnessLevel"]
    helpful = (levels == "HELPFUL").mean() * 100
    print(name, "|", len(levels), "ratings |", round(helpful), "% Helpful")

Note 1 | 113 ratings | 77 % Helpful
Note 2 | 101 ratings | 76 % Helpful
Note 3 | 77 ratings | 83 % Helpful
Note 4 | 165 ratings | 82 % Helpful


- All 4 notes got about 80% Helpful
- So votes alone don't explain why only Note 4 was shown

## 3. The raters
- `rater_modelingPopulation`: CORE or EXPANSION
- `rater_modelingGroup`: X's rater group (24, 13 are just ids)

In [9]:
# one row per rater
raters = ratings.drop_duplicates("raterParticipantId")

print("raters:", len(raters))
print("population:", raters["rater_modelingPopulation"].value_counts().to_dict())
print("biggest groups:", raters["rater_modelingGroup"].value_counts().head(2).to_dict())

raters: 263
population: {'EXPANSION': 204, 'CORE': 57, 'EXPANSION_PLUS': 2}
biggest groups: {24.0: 184, 13.0: 37}


In [10]:
# % Helpful from the 2 biggest groups
for name in note_names:
    this_note = ratings[ratings["note_label"] == name]
    group_24 = this_note[this_note["rater_modelingGroup"] == 24]
    group_13 = this_note[this_note["rater_modelingGroup"] == 13]
    print(name,
          "| group 24:", round((group_24["helpfulnessLevel"] == "HELPFUL").mean() * 100), "% Helpful",
          "| group 13:", round((group_13["helpfulnessLevel"] == "HELPFUL").mean() * 100), "% Helpful")

Note 1 | group 24: 92 % Helpful | group 13: 29 % Helpful
Note 2 | group 24: 91 % Helpful | group 13: 53 % Helpful
Note 3 | group 24: 97 % Helpful | group 13: 44 % Helpful
Note 4 | group 24: 96 % Helpful | group 13: 45 % Helpful


- Group 24 liked every note
- Group 13 was split

## 4. Trusted or not trusted
- Valid rating: Helpful or Not Helpful, within 48 hours, before X decided, on a decided note
- Trust score = matched X's decision / valid
- Trusted = trust score 0.66+ and 10+ ratings
- Reads all ratings files (about 3 minutes)

In [11]:
# every rating our raters ever made
rater_ids = set(ratings["raterParticipantId"])
keep_columns = ["noteId", "raterParticipantId", "createdAtMillis", "helpfulnessLevel", "helpful", "notHelpful"]

parts = []
for file in sorted(glob.glob("data/ratings/ratings-*.tsv")):
    for chunk in pd.read_csv(file, sep="\t", usecols=keep_columns, dtype=id_columns, chunksize=5_000_000):
        parts.append(chunk[chunk["raterParticipantId"].isin(rater_ids)])
history = pd.concat(parts)

print("past ratings:", len(history))

past ratings: 218395


In [12]:
# add X's final decision for each note they rated
status_columns = ["noteId", "createdAtMillis", "currentStatus", "timestampMillisOfFirstNonNMRStatus"]
status_file = pd.read_csv("data/noteStatusHistory-00000.tsv", sep="\t", usecols=status_columns, dtype=id_columns)
status_file = status_file.rename(columns={"createdAtMillis": "note_created_ms"})

history = history.merge(status_file, on="noteId", how="left")
history[["raterParticipantId", "helpfulnessLevel", "currentStatus"]].head()

,raterParticipantId,helpfulnessLevel,currentStatus
0,A808023961BF8E0BF10F7D6C49E51142999AEB6049030B7A4A7B66BE10C251BA,HELPFUL,CURRENTLY_RATED_HELPFUL
1,0955AA84D076DBBD3981330BF71BBD48096523477A9B60736AC9DD396E53BEE5,NOT_HELPFUL,NEEDS_MORE_RATINGS
2,C5113721E5B66F4DC13651FE323057861C4B73B51A822A882D016654CEC988C4,HELPFUL,NEEDS_MORE_RATINGS
3,FFDFC3C710E32529AA1BEDEC57D5D20D23864E1F2C72D6A6351242B938551F07,SOMEWHAT_HELPFUL,CURRENTLY_RATED_HELPFUL
4,0D26888A070A3A8C05430364D99E18967CF79B24DD28D28CADB4F5451A3721C2,NOT_HELPFUL,NEEDS_MORE_RATINGS


In [13]:
# rating as a number: 1 = Helpful, 0 = Not Helpful (old ratings use the helpful / notHelpful columns)
history["value"] = float("nan")
history.loc[(history["helpful"] == 1) | (history["helpfulnessLevel"] == "HELPFUL"), "value"] = 1
history.loc[(history["notHelpful"] == 1) | (history["helpfulnessLevel"] == "NOT_HELPFUL"), "value"] = 0

In [14]:
# X's 4 conditions for a valid rating
note_decided = history["currentStatus"].isin(["CURRENTLY_RATED_HELPFUL", "CURRENTLY_RATED_NOT_HELPFUL"])
within_48_hours = history["createdAtMillis"] - history["note_created_ms"] < 48 * 60 * 60 * 1000
before_decision = history["createdAtMillis"] < history["timestampMillisOfFirstNonNMRStatus"]
helpful_or_not = history["value"].isin([0, 1])

valid = history[note_decided & within_48_hours & before_decision & helpful_or_not].copy()
print("valid ratings:", len(valid))

valid ratings: 31916


In [15]:
# did the rating match X's final decision?
said_helpful_and_shown = (valid["value"] == 1) & (valid["currentStatus"] == "CURRENTLY_RATED_HELPFUL")
said_not_and_rejected = (valid["value"] == 0) & (valid["currentStatus"] == "CURRENTLY_RATED_NOT_HELPFUL")
valid["matched"] = said_helpful_and_shown | said_not_and_rejected

print("matched:", valid["matched"].sum())

matched: 27760


In [16]:
# one row per rater: counts and trust score
profile = raters.set_index("raterParticipantId")[["rater_modelingPopulation", "rater_modelingGroup"]].copy()
profile["total_ratings"] = history.groupby("raterParticipantId").size()
profile["valid_ratings"] = valid.groupby("raterParticipantId").size()
profile["agreeing_ratings"] = valid.groupby("raterParticipantId")["matched"].sum()
profile = profile.fillna({"valid_ratings": 0, "agreeing_ratings": 0})

profile["trust_score"] = (profile["agreeing_ratings"] / profile["valid_ratings"]).round(2)
profile["trusted"] = (profile["total_ratings"] >= 10) & (profile["trust_score"] >= 0.66)
profile.sample(5, random_state=1)

,rater_modelingPopulation,rater_modelingGroup,total_ratings,valid_ratings,agreeing_ratings,trust_score,trusted
raterParticipantId,,,,,,,
476ED34259534E557194F17F91E36217B5DEA39BD13F7C8B332DF06BCB80B479,EXPANSION,24.0,107,12.0,11.0,0.92,True
A59037968AA597CFFE41F43BF66785782197393F9053EE25999545B6EEC24AC5,CORE,13.0,1240,152.0,111.0,0.73,True
C8998F4E47BE5D5771CFE2F1DE41D89A6235A56BE1FEF74D6DCDDEAF02DB4406,EXPANSION,24.0,26,1.0,1.0,1.00,True
E21906CD42CD29031B5DAF1BF5A0BD052B43F52157EB324D06DFB302FCB32518,EXPANSION,27.0,43,8.0,8.0,1.00,True
38803E6524398B188C6CA75065532A23784FC2E708FD66FB7F177BF79725E2B8,EXPANSION,24.0,23,4.0,4.0,1.00,True


In [17]:
# save and count
profile.to_csv(saudi_folder + "rater_profiles.csv")

print("trusted:", profile["trusted"].sum())
print("not trusted:", (~profile["trusted"]).sum())

trusted: 227
not trusted: 36


In [18]:
# why the not-trusted raters were left out
not_trusted = profile[~profile["trusted"]]
few_ratings = not_trusted["total_ratings"] < 10
nothing_decided = ~few_ratings & (not_trusted["valid_ratings"] == 0)
low_trust = ~few_ratings & ~nothing_decided

print("under 10 ratings:", few_ratings.sum())
print("trust under 0.66:", low_trust.sum())
print("no rated note was decided by X:", nothing_decided.sum())

under 10 ratings: 17
trust under 0.66: 16
no rated note was decided by X: 3


- 227 trusted, 36 not trusted
- Saved in `rater_profiles.csv` (column `trusted`)

## 5. Which model decides
- X has a Core model and an Expansion model
- A model decides a note only if 50%+ of its ratings come from its groups

In [19]:
# Core groups, from X's code (constants.py)
core_groups = {1, 2, 3, 6, 8, 9, 10, 11, 13, 14, 19, 21, 25}

for name in note_names:
    groups = ratings.loc[ratings["note_label"] == name, "rater_modelingGroup"]
    core_share = groups.isin(core_groups).mean() * 100
    if core_share >= 50:
        model = "Core"
    else:
        model = "Expansion"
    print(name, "| from Core groups:", round(core_share), "% | decided by:", model)

Note 1 | from Core groups: 22 % | decided by: Expansion
Note 2 | from Core groups: 25 % | decided by: Expansion
Note 3 | from Core groups: 21 % | decided by: Expansion
Note 4 | from Core groups: 19 % | decided by: Expansion


In [20]:
# what X's data says
note_status[["note_label", "currentDecidedBy"]]

,note_label,currentDecidedBy
0,Note 1,ExpansionModel (v1.1)
1,Note 2,ExpansionModel (v1.1)
2,Note 3,ExpansionModel (v1.1)
3,Note 4,ExpansionModel (v1.1)


- Only 19-25% from Core groups, so Expansion decides
- Same as X's data

## 6. Each rater's side
- From `score_with_x_model.py` (X's own model, 1 month of ratings)
- `rater_side`: learned from who agrees with whom

In [21]:
# each rater's side
sides = pd.read_csv(saudi_folder + "rater_sides.csv", dtype=id_columns).set_index("raterParticipantId")
ratings["rater_side"] = ratings["raterParticipantId"].map(sides["rater_side"])

print("raters with a side:", ratings.drop_duplicates("raterParticipantId")["rater_side"].notna().sum())

raters with a side: 194


In [22]:
# average side of the 2 biggest groups
raters = ratings.drop_duplicates("raterParticipantId")
for group_id in [24, 13]:
    group_raters = raters[raters["rater_modelingGroup"] == group_id]
    print("group", group_id, "| average side:", round(group_raters["rater_side"].mean(), 2))

group 24 | average side: 0.39
group 13 | average side: -0.04


- Group 24 is on one side, group 13 is near the middle
- Main side = side above 0, other side = 0 or below

## 7. Each note's 2 numbers
- `helpfulness_score`: liked across sides (shown at 0.40+)
- `one_sidedness`: liked by one side only (blocked at 0.5+)

In [23]:
# scores from X's model
model_scores = pd.read_csv(saudi_folder + "model_scores.csv", dtype=id_columns)
model_scores["one_sided"] = model_scores["one_sidedness"].abs() >= 0.5

model_scores[["note_label", "writer_type", "helpfulness_score", "one_sidedness", "one_sided"]].round(2)

,note_label,writer_type,helpfulness_score,one_sidedness,one_sided
0,Note 1,Human,0.12,0.57,True
1,Note 2,AI,0.15,0.56,True
2,Note 3,AI,0.15,0.57,True
3,Note 4,Human,0.21,0.32,False


In [24]:
# ratings made before Note 4 was shown
note4_shown_ms = note_status.loc[note_status["note_label"] == "Note 4", "timestampMillisOfFirstNonNMRStatus"].iloc[0]
before_note4 = ratings[(ratings["createdAtMillis"] <= note4_shown_ms) & ratings["rater_side"].notna()]

print("ratings before Note 4 was shown:", len(before_note4))

ratings before Note 4 was shown: 279


In [25]:
# % Helpful from each side
for name in note_names:
    this_note = before_note4[before_note4["note_label"] == name]
    main_side = this_note[this_note["rater_side"] > 0]
    other_side = this_note[this_note["rater_side"] <= 0]
    print(name,
          "| main side:", round((main_side["helpfulnessLevel"] == "HELPFUL").mean() * 100), "% Helpful",
          "| other side:", round((other_side["helpfulnessLevel"] == "HELPFUL").mean() * 100), "% Helpful")

Note 1 | main side: 84 % Helpful | other side: 31 % Helpful
Note 2 | main side: 87 % Helpful | other side: 36 % Helpful
Note 3 | main side: 90 % Helpful | other side: 40 % Helpful
Note 4 | main side: 86 % Helpful | other side: 47 % Helpful


- Notes 1-3: one-sided (0.56-0.57)
- Note 4: 0.32, and the highest score (0.21)
- Main side liked all 4; other side liked Note 4 most (47% vs 31-40%)

## 8. X's decision

In [26]:
note_status[["note_label", "currentStatus", "currentExpansionStatus", "currentDecidedBy"]]

,note_label,currentStatus,currentExpansionStatus,currentDecidedBy
0,Note 1,NEEDS_MORE_RATINGS,FIRM_REJECT,ExpansionModel (v1.1)
1,Note 2,NEEDS_MORE_RATINGS,FIRM_REJECT,ExpansionModel (v1.1)
2,Note 3,NEEDS_MORE_RATINGS,NEEDS_MORE_RATINGS,ExpansionModel (v1.1)
3,Note 4,CURRENTLY_RATED_HELPFUL,CURRENTLY_RATED_HELPFUL,ExpansionModel (v1.1)


- Notes 1, 2: blocked (`FIRM_REJECT`)
- Note 3: still waiting (`NEEDS_MORE_RATINGS`)
- Note 4: shown, because both sides found it helpful
- My scores use 1 month of data, X uses more, so compare the notes with each other